# Model Selection V2 - Optimización de Hiperparámetros con Optuna

## Objetivo
Encontrar los mejores hiperparámetros de CatBoost para las 5 variables de alta importancia identificadas en el EDA.

## Variables Seleccionadas (del EDA)
1. `study_hours` (numérica)
2. `sleep_quality` (categórica)
3. `facility_rating` (categórica)
4. `class_attendance` (numérica)
5. `study_method` (categórica)

## Método
- **Algoritmo:** CatBoost
- **Optimización:** Optuna (Bayesian Optimization con pruning)
- **Validación:** 5-fold cross-validation (split 90/10)
- **Métrica:** RMSE promedio
- **Sample:** 50% de datos para optimización (modelo final con 100%)

## Tiempo Estimado
15-20 minutos

## 1. Setup e Imports

In [ ]:
# Imports
import pandas as pd
import numpy as np
import time
import joblib
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import optuna.visualization as vis
import warnings
warnings.filterwarnings('ignore')

# Configuración
RANDOM_SEED = 42
N_FOLDS = 5
N_TRIALS = 25  # Reducido para optimización más rápida
SAMPLE_FRACTION = 0.5  # 50% de datos para optimización

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Librerías cargadas")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ CV folds: {N_FOLDS}")
print(f"✓ Optuna trials: {N_TRIALS}")
print(f"✓ Sample fraction: {SAMPLE_FRACTION*100:.0f}%")

## 2. Carga de Datos y Selección de Variables

In [ ]:
# Ruta del archivo
DATA_PATH = r"C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores\data\train.csv"

# Cargar datos completos
print("Cargando datos...")
df = pd.read_csv(DATA_PATH)
print(f"✓ Dataset completo: {len(df):,} registros")

# ===== Sample del 50% para optimización más rápida =====
df_sample = df.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED)
print(f"✓ Sample para optimización: {len(df_sample):,} registros ({SAMPLE_FRACTION*100:.0f}%)")

# Variables seleccionadas del EDA (Top 5)
selected_features = [
    'study_hours',
    'sleep_quality',
    'facility_rating',
    'class_attendance',
    'study_method'
]

categorical_features = ['sleep_quality', 'facility_rating', 'study_method']

# Preparar datos CON SAMPLE (para optimización)
X = df_sample[selected_features]
y = df_sample['exam_score']

print(f"\n{'='*80}")
print("DATASET PREPARADO PARA OPTIMIZACIÓN")
print(f"{'='*80}")
print(f"Registros (sample): {len(df_sample):,}")
print(f"Registros (total):  {len(df):,}")
print(f"Variables seleccionadas: {len(selected_features)}")
print(f"  - Numéricas: {len([f for f in selected_features if f not in categorical_features])}")
print(f"  - Categóricas: {len(categorical_features)}")
print(f"Target: exam_score")
print(f"{'='*80}")

print(f"\nVariables:")
for i, feat in enumerate(selected_features, 1):
    feat_type = "CAT" if feat in categorical_features else "NUM"
    print(f"  {i}. {feat} [{feat_type}]")

print(f"\n⚠️ Nota: El modelo final se entrenará con 100% de los datos ({len(df):,} registros)")

## 3. Función de Cross-Validation

In [3]:
def cross_validate_catboost(X, y, cat_features, params, n_folds=5, random_state=42):
    """
    Cross-validation con K-Fold para CatBoost
    
    Parameters:
    -----------
    X : DataFrame con features
    y : Series con target
    cat_features : list de nombres de features categóricas
    params : dict con hiperparámetros de CatBoost
    n_folds : int, número de folds
    random_state : int, seed para reproducibilidad
    
    Returns:
    --------
    dict: mean_rmse, std_rmse, fold_rmses, total_time
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    fold_rmses = []
    fold_times = []
    
    # Identificar índices de features categóricas
    cat_indices = [i for i, col in enumerate(X.columns) if col in cat_features]
    
    for fold_num, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        start = time.time()
        
        # Crear y entrenar modelo
        model = CatBoostRegressor(**params, verbose=False)
        model.fit(
            X_train, y_train,
            cat_features=cat_indices,
            eval_set=(X_val, y_val),
            use_best_model=True,
            verbose=False
        )
        
        elapsed = time.time() - start
        
        # Predecir y calcular RMSE
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        fold_rmses.append(rmse)
        fold_times.append(elapsed)
    
    return {
        'mean_rmse': np.mean(fold_rmses),
        'std_rmse': np.std(fold_rmses),
        'fold_rmses': fold_rmses,
        'total_time': np.sum(fold_times)
    }

print("✓ Función de cross-validation definida")

✓ Función de cross-validation definida


## 4. Baseline - Modelo con Hiperparámetros por Defecto

In [ ]:
# Parámetros por defecto
baseline_params = {
    'loss_function': 'RMSE',
    'iterations': 50,  # Reducido para optimización rápida
    'depth': 6,
    'learning_rate': 0.1,
    'random_seed': RANDOM_SEED
}

print("="*80)
print("EVALUANDO BASELINE CON HIPERPARÁMETROS POR DEFECTO")
print("="*80)
print("Hiperparámetros:")
for key, value in baseline_params.items():
    print(f"  {key}: {value}")
print("="*80)

baseline_result = cross_validate_catboost(X, y, categorical_features, baseline_params, n_folds=N_FOLDS)

print(f"\n{'='*80}")
print("RESULTADOS DEL BASELINE")
print(f"{'='*80}")
print(f"RMSE promedio: {baseline_result['mean_rmse']:.6f}")
print(f"Desviación est: {baseline_result['std_rmse']:.6f}")
print(f"Tiempo total:   {baseline_result['total_time']:.2f}s ({baseline_result['total_time']/60:.2f} min)")
print(f"\nRMSE por fold:")
for i, rmse in enumerate(baseline_result['fold_rmses'], 1):
    print(f"  Fold {i}: {rmse:.6f}")
print(f"{'='*80}")

## 5. Espacio de Búsqueda de Hiperparámetros

In [ ]:
# Definir rangos para Optuna (reducidos para optimización rápida)
param_ranges = {
    'iterations': (30, 80),  # Reducido de (100, 500)
    'depth': (4, 10),
    'learning_rate': (0.01, 0.15),  # log scale
    'l2_leaf_reg': (1, 9),
    'subsample': (0.6, 1.0),
    'colsample_bylevel': (0.6, 1.0),
    'min_data_in_leaf': (1, 20)
}

print("\n" + "="*80)
print("ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS")
print("="*80)
for param, (min_val, max_val) in param_ranges.items():
    print(f"{param:20s}: [{min_val}, {max_val}]")
print("="*80)
print(f"\nMétodo: Bayesian Optimization (TPE)")
print(f"Pruning: Median Pruner (n_warmup_steps=5)")
print(f"Trials: {N_TRIALS}")
print(f"Sample: {SAMPLE_FRACTION*100:.0f}% de datos")
print("="*80)

## 6. Optimización con Optuna

In [ ]:
def objective(trial):
    """
    Función objetivo para Optuna
    
    Optuna aprende de trials anteriores y enfoca la búsqueda
    en regiones prometedoras del espacio de hiperparámetros.
    """
    # Sugerir hiperparámetros (rangos reducidos para optimización rápida)
    params = {
        'loss_function': 'RMSE',
        'iterations': trial.suggest_int('iterations', 30, 80),  # Reducido de 100-500
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 20),
        'random_seed': RANDOM_SEED
    }
    
    # Evaluar con cross-validation
    result = cross_validate_catboost(X, y, categorical_features, params, n_folds=N_FOLDS)
    
    # Optuna minimiza el valor de retorno
    return result['mean_rmse']

print("✓ Función objetivo de Optuna definida")

In [ ]:
# Crear estudio de Optuna
print("\n" + "="*80)
print("INICIANDO OPTIMIZACIÓN DE HIPERPARÁMETROS CON OPTUNA")
print("="*80)
print("Estrategia: Bayesian Optimization (TPE)")
print(f"Trials: {N_TRIALS}")
print(f"Sample: {SAMPLE_FRACTION*100:.0f}% de datos ({len(df_sample):,} registros)")
print("Pruning: Sí (descarta configuraciones malas early)")
print("Tiempo estimado: ~15-20 minutos")
print("="*80)

study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_SEED),  # Bayesian optimization
    pruner=MedianPruner(n_warmup_steps=5)  # Descarta trials malos después de 5 folds
)

# Optimizar
print("\nIniciando optimización...\n")
start_time = time.time()

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

optimization_time = time.time() - start_time

print(f"\n{'='*80}")
print("OPTIMIZACIÓN COMPLETADA")
print(f"{'='*80}")
print(f"Tiempo total: {optimization_time/60:.2f} minutos ({optimization_time/3600:.2f} horas)")
print(f"{'='*80}")

## 7. Análisis de Resultados de Optimización

In [ ]:
# Mejores hiperparámetros
print("\n" + "="*80)
print("RESULTADOS DE OPTIMIZACIÓN")
print("="*80)
print("Mejores hiperparámetros encontrados:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {key:20s}: {value:.6f}")
    else:
        print(f"  {key:20s}: {value}")

print(f"\nMejor RMSE (CV):       {study.best_value:.6f}")
print(f"Trials completados:    {len(study.trials)}")
print(f"Trials podados:        {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"Trials completados:    {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print("="*80)

In [ ]:
# Comparación con baseline
print("\n" + "="*80)
print("COMPARACIÓN: BASELINE vs OPTIMIZADO")
print("="*80)
print(f"Baseline RMSE:        {baseline_result['mean_rmse']:.6f} ± {baseline_result['std_rmse']:.6f}")
print(f"Optimizado RMSE:      {study.best_value:.6f}")
print(f"\nMejora absoluta:      {baseline_result['mean_rmse'] - study.best_value:.6f}")
print(f"Mejora porcentual:    {((baseline_result['mean_rmse'] - study.best_value) / baseline_result['mean_rmse'] * 100):.2f}%")
print(f"\nTrials ejecutados:    {len(study.trials)}")
print(f"Trials podados:       {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])} ({len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])/len(study.trials)*100:.1f}%)")
print("="*80)

In [ ]:
# Top 10 mejores configuraciones
trials_df = study.trials_dataframe()
trials_df_sorted = trials_df.sort_values('value')

print("\n" + "="*80)
print("TOP 10 MEJORES CONFIGURACIONES")
print("="*80)
display_cols = ['number', 'value', 'params_iterations', 'params_depth', 'params_learning_rate', 'state']
print(trials_df_sorted[display_cols].head(10).to_string(index=False))
print("="*80)

## 8. Visualizaciones de Optuna

In [ ]:
# 1. Historia de optimización
fig = vis.plot_optimization_history(study)
fig.write_image('optuna_optimization_history.png')
fig.show()
print("✓ Historia guardada en 'optuna_optimization_history.png'")

In [ ]:
# 2. Importancia de hiperparámetros
fig = vis.plot_param_importances(study)
fig.write_image('optuna_param_importances.png')
fig.show()
print("✓ Importancia de parámetros guardada en 'optuna_param_importances.png'")

In [ ]:
# 3. Slice plot (relación de cada parámetro con RMSE)
fig = vis.plot_slice(study)
fig.write_image('optuna_slice_plot.png')
fig.show()
print("✓ Slice plot guardado en 'optuna_slice_plot.png'")

In [ ]:
# 4. Parallel coordinate plot
fig = vis.plot_parallel_coordinate(study)
fig.write_image('optuna_parallel_coordinate.png')
fig.show()
print("✓ Parallel coordinate guardado en 'optuna_parallel_coordinate.png'")

## 9. Entrenar Modelo Final con Mejores Hiperparámetros

In [ ]:
# Mejores parámetros de Optuna
best_params = study.best_params.copy()
best_params['loss_function'] = 'RMSE'
best_params['random_seed'] = RANDOM_SEED
best_params['verbose'] = True

print("\n" + "="*80)
print("ENTRENANDO MODELO FINAL CON MEJORES HIPERPARÁMETROS")
print("="*80)
print("Configuración óptima:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {key:20s}: {value:.6f}")
    else:
        print(f"  {key:20s}: {value}")
print("="*80)

# ===== IMPORTANTE: Entrenar en 100% de los datos (no en sample) =====
X_full = df[selected_features]
y_full = df['exam_score']

cat_indices = [i for i, col in enumerate(X_full.columns) if col in categorical_features]

print(f"\n⚠️ Entrenando modelo final con 100% de los datos ({len(df):,} registros)...")
print(f"   (La optimización usó {SAMPLE_FRACTION*100:.0f}% = {len(df_sample):,} registros)")

final_model = CatBoostRegressor(**best_params)
final_model.fit(X_full, y_full, cat_features=cat_indices)

print(f"\n✓ Modelo final entrenado con 100% datos ({len(df):,} registros)")
print(f"✓ RMSE esperado en CV (sample): {study.best_value:.6f}")

## 10. Feature Importance del Modelo Final

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "="*60)
print("FEATURE IMPORTANCE DEL MODELO FINAL")
print("="*60)
print(importance_df.to_string(index=False))
print("="*60)

In [ ]:
# Visualizar feature importance
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue', edgecolor='black')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Feature Importance - Modelo Optimizado con Optuna', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('final_model_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Feature importance guardado en 'final_model_feature_importance.png'")

## 11. Validación del Modelo

In [ ]:
# Predecir en datos de entrenamiento completos (sanity check)
y_pred_train = final_model.predict(X_full)
rmse_train = np.sqrt(mean_squared_error(y_full, y_pred_train))

print("\n" + "="*80)
print("VALIDACIÓN DEL MODELO FINAL")
print("="*80)
print(f"RMSE en train (100% datos): {rmse_train:.6f}")
print(f"RMSE esperado en CV (50% sample): {study.best_value:.6f}")
print(f"Diferencia: {study.best_value - rmse_train:.6f}")
print("\nNota: RMSE en train debería ser menor que CV (el modelo")
print("      se ajusta mejor a datos que ya ha visto)")
print("="*80)

# Validación
if rmse_train < study.best_value:
    print("\n✓ Validación exitosa: RMSE train < RMSE CV")
else:
    print("\n⚠ Advertencia: RMSE train >= RMSE CV (esto puede ocurrir porque")
    print("   el CV se hizo con 50% de datos y el train con 100%)")

## 12. Exportación de Resultados

In [ ]:
print("\n" + "="*80)
print("EXPORTANDO RESULTADOS")
print("="*80)

# 1. Guardar modelo
final_model.save_model('best_catboost_model_v2.cbm')
print("✓ Modelo guardado en 'best_catboost_model_v2.cbm'")

# 2. Guardar hiperparámetros
with open('best_hyperparameters.txt', 'w') as f:
    f.write("MEJORES HIPERPARÁMETROS - CatBoost Model V2 (Optimizado con Optuna)\n")
    f.write("="*70 + "\n\n")
    f.write(f"RMSE (CV 5-fold): {study.best_value:.6f}\n")
    f.write(f"RMSE (Train):     {rmse_train:.6f}\n")
    f.write(f"Trials completados: {len(study.trials)}\n")
    f.write(f"Trials podados: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}\n\n")
    f.write("Hiperparámetros:\n")
    for key, value in study.best_params.items():
        f.write(f"  {key}: {value}\n")
    f.write("\nVariables utilizadas:\n")
    for var in selected_features:
        f.write(f"  - {var}\n")
    f.write("\nMétodo de optimización: Bayesian Optimization (TPE) con Median Pruner\n")
    f.write(f"Tiempo de optimización: {optimization_time/60:.2f} minutos\n")

print("✓ Hiperparámetros guardados en 'best_hyperparameters.txt'")

# 3. Guardar estudio de Optuna
joblib.dump(study, 'optuna_study.pkl')
print("✓ Estudio de Optuna guardado en 'optuna_study.pkl'")

# 4. Guardar todos los trials
trials_df.to_csv('hyperparameter_search_results.csv', index=False)
print("✓ Resultados de búsqueda guardados en 'hyperparameter_search_results.csv'")

# 5. Guardar trials completos
trials_df.to_csv('optuna_trials_complete.csv', index=False)
print("✓ Trials completos guardados en 'optuna_trials_complete.csv'")

# 6. Guardar feature importance
importance_df.to_csv('feature_importance_final.csv', index=False)
print("✓ Feature importance guardada en 'feature_importance_final.csv'")

print("="*80)
print("✓ Todos los resultados exportados correctamente")
print("="*80)

## 13. Resumen Final

In [ ]:
print("\n" + "="*100)
print("RESUMEN FINAL - OPTIMIZACIÓN DE HIPERPARÁMETROS")
print("="*100)

print("\n📊 DATASET:")
print(f"  - Registros totales: {len(df):,}")
print(f"  - Sample para optimización: {len(df_sample):,} ({SAMPLE_FRACTION*100:.0f}%)")
print(f"  - Variables seleccionadas: {len(selected_features)}")
print(f"  - Numéricas: {len([f for f in selected_features if f not in categorical_features])}")
print(f"  - Categóricas: {len(categorical_features)}")

print("\n🔍 OPTIMIZACIÓN:")
print(f"  - Método: Bayesian Optimization (TPE) con Median Pruner")
print(f"  - Trials ejecutados: {len(study.trials)}")
print(f"  - Trials podados: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"  - Tiempo total: {optimization_time/60:.2f} min ({optimization_time/3600:.2f} h)")

print("\n🏆 RESULTADOS:")
print(f"  - Baseline RMSE:        {baseline_result['mean_rmse']:.6f}")
print(f"  - Mejor RMSE (CV):      {study.best_value:.6f}")
print(f"  - RMSE (Train 100%):    {rmse_train:.6f}")
print(f"  - Mejora vs baseline:   {baseline_result['mean_rmse'] - study.best_value:.6f} ({((baseline_result['mean_rmse'] - study.best_value) / baseline_result['mean_rmse'] * 100):.2f}%)")

print("\n⚙️ MEJORES HIPERPARÁMETROS:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  - {key:20s}: {value:.6f}")
    else:
        print(f"  - {key:20s}: {value}")

print("\n📈 FEATURE IMPORTANCE:")
for i, (_, row) in enumerate(importance_df.iterrows(), 1):
    print(f"  {i}. {row['feature']:20s}: {row['importance']:.4f}")

print("\n💾 ARCHIVOS GENERADOS:")
print("  - best_catboost_model_v2.cbm")
print("  - best_hyperparameters.txt")
print("  - optuna_study.pkl")
print("  - hyperparameter_search_results.csv")
print("  - optuna_trials_complete.csv")
print("  - feature_importance_final.csv")
print("  - optuna_optimization_history.png")
print("  - optuna_param_importances.png")
print("  - optuna_slice_plot.png")
print("  - optuna_parallel_coordinate.png")
print("  - final_model_feature_importance.png")

print("\n📝 PRÓXIMOS PASOS:")
print("  1. Cargar test.csv y generar predicciones")
print("  2. Crear submission para Kaggle")
print("  3. Considerar ensemble con diferentes seeds")
print("  4. Explorar feature engineering con las 5 variables")
print("  5. Probar agregar 1-2 variables adicionales")

print("\n" + "="*100)
print("✅ OPTIMIZACIÓN DE HIPERPARÁMETROS COMPLETADA EXITOSAMENTE")
print("="*100)